In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import arff
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42


In [ ]:
try:
    raw_data, _ = arff.loadarff('bank.arff')
except FileNotFoundError:
    raw_data, _ = arff.loadarff('project/bank.arff')

df = pd.DataFrame(raw_data)

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.decode('utf-8')

df = df.rename(columns={
    'V1': 'age', 'V2': 'job', 'V3': 'marital', 'V4': 'education',
    'V5': 'default', 'V6': 'balance', 'V7': 'housing', 'V8': 'loan',
    'V9': 'contact', 'V10': 'day', 'V11': 'month', 'V12': 'duration',
    'V13': 'campaign', 'V14': 'pdays', 'V15': 'previous', 'V16': 'poutcome',
    'Class': 'y',
})

df['y'] = df['y'].map({'1': 'no', '2': 'yes'})

cat_cols = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

for col in cat_cols + ['y']:
    df[col] = df[col].astype('category')

df[num_cols] = df[num_cols].astype('int64')

print(df.shape)
print('доля yes:', (df['y'] == 'yes').mean())
df.head()


In [ ]:
df.info()


In [ ]:
print('пропуски:')
print(df.isna().sum())
print('дубликаты:', df.duplicated().sum())

df['y'].value_counts()


In [ ]:
df['y'].value_counts().plot(kind='bar')
plt.show()

month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
               'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

yes_by_month = df.groupby('month', observed=True)['y'].apply(lambda s: (s == 'yes').mean())
yes_by_month = yes_by_month.reindex(month_order)
yes_by_month.plot(kind='bar')
plt.show()


In [ ]:
df[num_cols].hist(bins=30, figsize=(14, 8))
plt.tight_layout()
plt.show()


In [ ]:
for col in cat_cols:
    print(col)
    print(df[col].value_counts())
    print()


In [ ]:
for col in cat_cols:
    rates = df.groupby(col, observed=True)['y'].apply(lambda s: (s == 'yes').mean())
    if col == 'month':
        rates = rates.reindex(month_order)
    rates.plot(kind='bar', title=col)
    plt.show()


In [ ]:
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f')
plt.show()
corr


In [ ]:
for col in num_cols:
    df.boxplot(column=col, by='y', showfliers=False)
    plt.suptitle('')
    plt.show()


In [ ]:
df = df.drop(columns=['duration'])

df['was_contacted_before'] = (df['pdays'] != -1).astype(int)
df.loc[df['pdays'] == -1, 'pdays'] = 0
df['y_bin'] = (df['y'] == 'yes').astype(int)

cat_features = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
num_features = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'was_contacted_before']

print(df.shape)
print('доля yes:', df['y_bin'].mean())
df[cat_features + num_features + ['y_bin']].head()


In [ ]:
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print('train:', train_df.shape, 'yes=', train_df['y_bin'].mean())
print('val:  ', val_df.shape, 'yes=', val_df['y_bin'].mean())
print('test: ', test_df.shape, 'yes=', test_df['y_bin'].mean())


In [ ]:
def score_prep(scaler, use_was_contacted):
    nums = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
    if use_was_contacted:
        nums = nums + ['was_contacted_before']

    cols = cat_features + nums
    X_tr = train_df[cols]
    y_tr = train_df['y_bin']
    X_va = val_df[cols]
    y_va = val_df['y_bin']

    pipe = Pipeline([
        ('pre', ColumnTransformer([
            ('num', scaler, nums),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ])),
        ('model', LogisticRegression(
            max_iter=10000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    return average_precision_score(y_va, proba)


ablation = pd.DataFrame([
    {'scaler': 'StandardScaler', 'was_contacted_before': 'да',
     'PR-AUC (val)': score_prep(StandardScaler(), True)},
    {'scaler': 'RobustScaler', 'was_contacted_before': 'да',
     'PR-AUC (val)': score_prep(RobustScaler(), True)},
    {'scaler': 'StandardScaler', 'was_contacted_before': 'нет',
     'PR-AUC (val)': score_prep(StandardScaler(), False)},
    {'scaler': 'RobustScaler', 'was_contacted_before': 'нет',
     'PR-AUC (val)': score_prep(RobustScaler(), False)},
])
ablation = ablation.sort_values('PR-AUC (val)', ascending=False).reset_index(drop=True)
ablation


In [ ]:
best = ablation.iloc[0]
use_was = best['was_contacted_before'] == 'да'
scaler_name = best['scaler']
print(best)

if use_was:
    num_features = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'was_contacted_before']
else:
    num_features = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

feature_cols = cat_features + num_features

X_train = train_df[feature_cols]
y_train = train_df['y_bin']
X_val = val_df[feature_cols]
y_val = val_df['y_bin']
X_test = test_df[feature_cols]
y_test = test_df['y_bin']


In [ ]:
def make_scaler():
    if scaler_name == 'RobustScaler':
        return RobustScaler()
    return StandardScaler()


def get_metrics(model, X, y):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'roc_auc': roc_auc_score(y, proba),
        'pr_auc': average_precision_score(y, proba),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
    }


def fit_logreg(name, model, param_grid=None):
    pipe = Pipeline([
        ('pre', ColumnTransformer([
            ('num', make_scaler(), num_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ])),
        ('model', model),
    ])

    if param_grid is None:
        pipe.fit(X_train, y_train)
        best_pipe = pipe
        best_params = {}
    else:
        grid = GridSearchCV(pipe, param_grid, scoring='average_precision', cv=3, n_jobs=-1)
        grid.fit(X_train, y_train)
        best_pipe = grid.best_estimator_
        best_params = grid.best_params_

    val_m = get_metrics(best_pipe, X_val, y_val)
    test_m = get_metrics(best_pipe, X_test, y_test)
    return {
        'model': name,
        'best_params': best_params,
        'PR-AUC (val)': val_m['pr_auc'],
        'ROC-AUC (val)': val_m['roc_auc'],
        'F1 (val)': val_m['f1'],
        'PR-AUC (test)': test_m['pr_auc'],
        'ROC-AUC (test)': test_m['roc_auc'],
        'F1 (test)': test_m['f1'],
        'fitted': best_pipe,
    }


In [ ]:
C_GRID = np.logspace(-3, 2, 8)

results = []

results.append(fit_logreg(
    'LogReg (none)',
    LogisticRegression(
        penalty=None, solver='lbfgs', max_iter=10000,
        class_weight='balanced', random_state=RANDOM_STATE,
    ),
))

results.append(fit_logreg(
    'LogReg L2',
    LogisticRegression(
        penalty='l2', solver='lbfgs', max_iter=10000,
        class_weight='balanced', random_state=RANDOM_STATE,
    ),
    param_grid={'model__C': C_GRID},
))

results.append(fit_logreg(
    'LogReg L1',
    LogisticRegression(
        penalty='l1', solver='saga', max_iter=10000,
        class_weight='balanced', random_state=RANDOM_STATE,
    ),
    param_grid={'model__C': C_GRID},
))

results.append(fit_logreg(
    'LogReg ElasticNet',
    LogisticRegression(
        penalty='elasticnet', solver='saga', max_iter=10000,
        class_weight='balanced', random_state=RANDOM_STATE,
    ),
    param_grid={'model__C': C_GRID, 'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
))

metrics_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('fitted', 'best_params')} for r in results
])
metrics_df = metrics_df.sort_values('PR-AUC (val)', ascending=False).reset_index(drop=True)
metrics_df


In [ ]:
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()
X_test_cb = X_test.copy()

for col in cat_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_val_cb[col] = X_val_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

best_cb = None
best_cb_score = -1

for depth in [4, 6]:
    for lr in [0.03, 0.05, 0.1]:
        for l2 in [1, 3, 10]:
            model = CatBoostClassifier(
                loss_function='Logloss',
                auto_class_weights='Balanced',
                random_seed=RANDOM_STATE,
                verbose=False,
                iterations=500,
                depth=depth,
                learning_rate=lr,
                l2_leaf_reg=l2,
                od_type='Iter',
                od_wait=50,
            )
            model.fit(
                X_train_cb, y_train,
                cat_features=cat_features,
                eval_set=(X_val_cb, y_val),
                use_best_model=True,
            )
            val_m = get_metrics(model, X_val_cb, y_val)
            test_m = get_metrics(model, X_test_cb, y_test)
            if val_m['pr_auc'] > best_cb_score:
                best_cb_score = val_m['pr_auc']
                best_cb = {
                    'model': 'CatBoost',
                    'best_params': {
                        'depth': depth,
                        'learning_rate': lr,
                        'l2_leaf_reg': l2,
                        'best_iteration': model.get_best_iteration(),
                    },
                    'PR-AUC (val)': val_m['pr_auc'],
                    'ROC-AUC (val)': val_m['roc_auc'],
                    'F1 (val)': val_m['f1'],
                    'PR-AUC (test)': test_m['pr_auc'],
                    'ROC-AUC (test)': test_m['roc_auc'],
                    'F1 (test)': test_m['f1'],
                    'fitted': model,
                    'is_catboost': True,
                }

results.append(best_cb)

metrics_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('fitted', 'best_params', 'is_catboost')} for r in results
])
metrics_df = metrics_df.sort_values('PR-AUC (val)', ascending=False).reset_index(drop=True)
metrics_df


In [ ]:
for r in results:
    print(r['model'], r['best_params'])


In [ ]:
metrics_df.set_index('model')[['PR-AUC (test)', 'ROC-AUC (test)', 'F1 (test)']].plot(kind='bar')
plt.show()


In [ ]:
best_result = max(results, key=lambda r: r['PR-AUC (val)'])
final_model = best_result['fitted']
final_name = best_result['model']
is_catboost = best_result.get('is_catboost', False)

print('final:', final_name)
print('PR-AUC val:', best_result['PR-AUC (val)'])
print('PR-AUC test:', best_result['PR-AUC (test)'])
print('params:', best_result['best_params'])

if is_catboost:
    X_test_final = X_test_cb
    X_val_final = X_val_cb
    X_train_final = X_train_cb
    importance = pd.Series(final_model.get_feature_importance(), index=feature_cols)
else:
    X_test_final = X_test
    X_val_final = X_val
    X_train_final = X_train
    pre = final_model.named_steps['pre']
    names = list(num_features) + list(pre.named_transformers_['cat'].get_feature_names_out(cat_features))
    coefs = np.abs(final_model.named_steps['model'].coef_.ravel())
    importance = pd.Series(coefs, index=names)

importance = importance.sort_values(ascending=False)
importance.head(15)


In [ ]:
importance.head(15).sort_values().plot(kind='barh')
plt.show()


In [ ]:
y_test_proba = final_model.predict_proba(X_test_final)[:, 1]
y_test_pred = (y_test_proba >= 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], '--')
plt.title('ROC-AUC = %.3f' % roc_auc_score(y_test, y_test_proba))
plt.show()

print(classification_report(y_test, y_test_pred, target_names=['no', 'yes']))
print(confusion_matrix(y_test, y_test_pred))


In [ ]:
rows = []
for name, X, y in [('train', X_train_final, y_train),
                   ('val', X_val_final, y_val),
                   ('test', X_test_final, y_test)]:
    m = get_metrics(final_model, X, y)
    m['split'] = name
    rows.append(m)

metrics_by_split = pd.DataFrame(rows).set_index('split')
print('ROC-AUC train-val:', metrics_by_split.loc['train', 'roc_auc'] - metrics_by_split.loc['val', 'roc_auc'])
print('ROC-AUC train-test:', metrics_by_split.loc['train', 'roc_auc'] - metrics_by_split.loc['test', 'roc_auc'])
metrics_by_split
